# **Database Migration from SQL Server from PostgreSQL**

In [1]:
import os
import pandas as pd 
import pyodbc
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import urllib
from IPython.display import display
from collections import defaultdict, deque
import warnings

### 1. Load Credentials

In [2]:
load_dotenv()

True

In [3]:
sql_host = os.getenv("SQL_SERVER_HOST")
sql_db = os.getenv("SQL_SERVER_DB")
sql_user = os.getenv("SQL_SERVER_USER")
sql_password = os.getenv("SQL_SERVER_PASSWORD")

# PostgreSQL
pg_host = os.getenv("POSTGRES_HOST")
pg_port = os.getenv("POSTGRES_PORT")
pg_db = os.getenv("POSTGRES_DB")
pg_user = os.getenv("POSTGRES_USER")
pg_password = os.getenv("POSTGRES_PASSWORD")

# ODBC Driver
driver = '{ODBC Driver 17 for SQL Server}'

### 2. Connect to SQL Server

In [4]:
print("Connecting to SQL Server ...")
print(f"     Server: {sql_host}")
print(f"     Database: {sql_db}")

Connecting to SQL Server ...
     Server: 127.0.0.1,1433
     Database: AdventureWorks2025


In [5]:
# Build connection string for SQLAlchemy
try: 
    sql_connection_string = urllib.parse.quote_plus(
        f'DRIVER={driver};'
        f'SERVER={sql_host};'
        f'DATABASE={sql_db};'
        f'UID={sql_user};'
        f'PWD={sql_password};'
        f'Encrypt=yes;'
        f'TrustServerCertificate=yes;'
    )
    sql_connection_engine = create_engine(f"mssql+pyodbc:///?odbc_connect={sql_connection_string}")
    print("[SUCCESS] -> Connection to SQL Server now live.")
except SQLAlchemyError as e:
    print(f"SQL Server connection failed: {e}")
    print(""" How to troubleshoot:
            > 1. Check credentials in .env file are correct
            > 2. Verify SQL Server is running
            > 3. Check Windows Authentification is enable or SQL Login on macOS
            > 4. Check that Encryption is Madatory for connection            
          """)

[SUCCESS] -> Connection to SQL Server now live.


### 3. Connect to PostgreSQL

In [6]:
print("Connecting to PostgreSQL ...")
print(f"     Server: {pg_host}")
print(f"     Database: {pg_db}")

Connecting to PostgreSQL ...
     Server: 127.0.0.1
     Database: AdventureWorks2025_UAT


In [7]:
try:
    pg_connection_engine = create_engine(
        f"postgresql+psycopg2://{pg_user}:{pg_password}"
        f"@{pg_host}:{pg_port}/{pg_db}",
        connect_args={
            "host" : pg_host,
            "port" : pg_port
        }
    )
    # Test connection và check version
    with pg_connection_engine.connect() as conn:
        result = conn.execute(text("SELECT version()"))
        version = result.fetchone()[0]
    
    print("Connected to PostgreSQL")
    print(f"     Version: {version[:50]}...\n")

except SQLAlchemyError as e:
        print(f"PostgreSQL connection failed: {e}")
        print(""" How to troubleshoot:
          > 1. Check PostgreSQL is running
          > 2. Verify username and password
          > 3. Check database exists
          """)
except Exception as e: 
    print(f"Unexpected errors: {e}")
    raise

Connected to PostgreSQL
     Version: PostgreSQL 18.3 (Homebrew) on aarch64-apple-darwin...



### 4. Migration Dependency Analysis

#### 4.1 Database Schema Exploration
#####  Tables and row counts


In [8]:
try:
    query = """
    SELECT 
        s.name AS schema_name,
        t.name AS table_name,
        s.name + '.' + t.name AS full_name,
        SUM(p.rows) AS row_count
    FROM sys.tables t
    JOIN sys.schemas s ON t.schema_id = s.schema_id
    JOIN sys.partitions p ON t.object_id = p.object_id
    WHERE p.index_id IN (0, 1)
    GROUP BY s.name, t.name
    ORDER BY s.name, t.name;
    """

    df_tables = pd.read_sql(query, sql_connection_engine)
    #print(f"Total tables: {len(df_tables)}")
    #display(df_tables.head(20))
except Exception as e:
    print(f"Query failed: {e}")

/opt/anaconda3/envs/database-migration-pipeline/lib/python3.11/site-packages/pandas/io/sql.py:1649: SAWarning: Unrecognized server version info '17.0.4015.4'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


In [9]:
print(f"Total tables: {len(df_tables)}")
print(f"Total rows across all tables: {df_tables['row_count'].sum():,} rows")
display(df_tables.head(20))

Total tables: 71
Total rows across all tables: 760,167 rows


,schema_name,table_name,full_name,row_count
0,dbo,AWBuildVersion,dbo.AWBuildVersion,1
1,dbo,DatabaseLog,dbo.DatabaseLog,927
2,dbo,ErrorLog,dbo.ErrorLog,0
3,HumanResources,Department,HumanResources.Department,16
4,HumanResources,Employee,HumanResources.Employee,290
5,HumanResources,EmployeeDepartmentHistory,HumanResources.EmployeeDepartmentHistory,296
6,HumanResources,EmployeePayHistory,HumanResources.EmployeePayHistory,316
7,HumanResources,JobCandidate,HumanResources.JobCandidate,13
8,HumanResources,Shift,HumanResources.Shift,3
9,Person,Address,Person.Address,19614


##### Foreign keys

In [10]:
try:
    query = """
    SELECT 
        s.name AS schema_name,
        t.name AS table_name,
        fk.name AS foreign_key_name,
        c.name AS fk_column,
        rs.name AS referenced_schema,
        rt.name AS referenced_table,
        rc.name AS referenced_column
    FROM sys.foreign_keys fk
    JOIN sys.tables t 
        ON fk.parent_object_id = t.object_id
    JOIN sys.schemas s 
        ON t.schema_id = s.schema_id
    JOIN sys.foreign_key_columns fkc 
        ON fk.object_id = fkc.constraint_object_id
    JOIN sys.columns c 
        ON fkc.parent_object_id = c.object_id 
        AND fkc.parent_column_id = c.column_id
    JOIN sys.tables rt 
        ON fk.referenced_object_id = rt.object_id
    JOIN sys.schemas rs 
        ON rt.schema_id = rs.schema_id
    JOIN sys.columns rc 
        ON fkc.referenced_object_id = rc.object_id 
        AND fkc.referenced_column_id = rc.column_id
    ORDER BY s.name, t.name;
    """

    df_fk = pd.read_sql(query, sql_connection_engine)
    print(f"Total tables: {len(df_fk)}")
    display(df_fk.head(20))
except Exception as e:
    print(f"Query failed: {e}")

Total tables: 91


,schema_name,table_name,foreign_key_name,fk_column,referenced_schema,referenced_table,referenced_column
0,HumanResources,Employee,FK_Employee_Person_BusinessEntityID,BusinessEntityID,Person,Person,BusinessEntityID
1,HumanResources,EmployeeDepartmentHistory,FK_EmployeeDepartmentHistory_Department_Depart...,DepartmentID,HumanResources,Department,DepartmentID
2,HumanResources,EmployeeDepartmentHistory,FK_EmployeeDepartmentHistory_Employee_Business...,BusinessEntityID,HumanResources,Employee,BusinessEntityID
3,HumanResources,EmployeeDepartmentHistory,FK_EmployeeDepartmentHistory_Shift_ShiftID,ShiftID,HumanResources,Shift,ShiftID
4,HumanResources,EmployeePayHistory,FK_EmployeePayHistory_Employee_BusinessEntityID,BusinessEntityID,HumanResources,Employee,BusinessEntityID
5,HumanResources,JobCandidate,FK_JobCandidate_Employee_BusinessEntityID,BusinessEntityID,HumanResources,Employee,BusinessEntityID
6,Person,Address,FK_Address_StateProvince_StateProvinceID,StateProvinceID,Person,StateProvince,StateProvinceID
7,Person,BusinessEntityAddress,FK_BusinessEntityAddress_Address_AddressID,AddressID,Person,Address,AddressID
8,Person,BusinessEntityAddress,FK_BusinessEntityAddress_AddressType_AddressTy...,AddressTypeID,Person,AddressType,AddressTypeID
9,Person,BusinessEntityAddress,FK_BusinessEntityAddress_BusinessEntity_Busine...,BusinessEntityID,Person,BusinessEntity,BusinessEntityID


##### Primary Key

In [11]:
try:
    query_pk = """
    SELECT 
        s.name AS schema_name,
        t.name AS table_name,
        c.name AS pk_column,
        ic.key_ordinal AS key_order,
        tp.name AS data_type
    FROM sys.tables t
    JOIN sys.schemas s 
        ON t.schema_id = s.schema_id
    JOIN sys.indexes i 
        ON t.object_id = i.object_id
        AND i.is_primary_key = 1
    JOIN sys.index_columns ic 
        ON i.object_id = ic.object_id
        AND i.index_id = ic.index_id
    JOIN sys.columns c 
        ON ic.object_id = c.object_id
        AND ic.column_id = c.column_id
    JOIN sys.types tp 
        ON c.user_type_id = tp.user_type_id
    ORDER BY s.name, t.name, ic.key_ordinal
    """
    df_pk = pd.read_sql(query_pk, sql_connection_engine)
    print(f"Total PK columns found: {len(df_pk)}")
    display(df_pk.head(20))
except Exception as e:
    print(f"Query failed: {e}")

Total PK columns found: 102


,schema_name,table_name,pk_column,key_order,data_type
0,dbo,AWBuildVersion,SystemInformationID,1,tinyint
1,dbo,DatabaseLog,DatabaseLogID,1,int
2,dbo,ErrorLog,ErrorLogID,1,int
3,HumanResources,Department,DepartmentID,1,smallint
4,HumanResources,Employee,BusinessEntityID,1,int
5,HumanResources,EmployeeDepartmentHistory,BusinessEntityID,1,int
6,HumanResources,EmployeeDepartmentHistory,StartDate,2,date
7,HumanResources,EmployeeDepartmentHistory,DepartmentID,3,smallint
8,HumanResources,EmployeeDepartmentHistory,ShiftID,4,tinyint
9,HumanResources,EmployeePayHistory,BusinessEntityID,1,int


##### Data Types each Column

In [12]:
try:
    query_dtypes = """
    SELECT 
        s.name AS schema_name,
        t.name AS table_name,
        c.name AS column_name,
        c.column_id AS column_order,
        tp.name AS data_type,
        c.max_length,
        c.precision,
        c.scale,
        c.is_nullable,
        c.is_identity
    FROM sys.tables t
    JOIN sys.schemas s 
        ON t.schema_id = s.schema_id
    JOIN sys.columns c 
        ON t.object_id = c.object_id
    JOIN sys.types tp 
        ON c.user_type_id = tp.user_type_id
    ORDER BY s.name, t.name, c.column_id
    """
    df_dtypes = pd.read_sql(query_dtypes, sql_connection_engine)
    print(f"Total columns found: {len(df_dtypes)}")
    display(df_dtypes.head(20))
except Exception as e:
    print(f"Query failed: {e}")

Total columns found: 486


,schema_name,table_name,column_name,column_order,data_type,max_length,precision,scale,is_nullable,is_identity
0,dbo,AWBuildVersion,SystemInformationID,1,tinyint,1,3,0,False,True
1,dbo,AWBuildVersion,Database Version,2,nvarchar,50,0,0,False,False
2,dbo,AWBuildVersion,VersionDate,3,datetime,8,23,3,False,False
3,dbo,AWBuildVersion,ModifiedDate,4,datetime,8,23,3,False,False
4,dbo,DatabaseLog,DatabaseLogID,1,int,4,10,0,False,True
5,dbo,DatabaseLog,PostTime,2,datetime,8,23,3,False,False
6,dbo,DatabaseLog,DatabaseUser,3,sysname,256,0,0,False,False
7,dbo,DatabaseLog,Event,4,sysname,256,0,0,False,False
8,dbo,DatabaseLog,Schema,5,sysname,256,0,0,True,False
9,dbo,DatabaseLog,Object,6,sysname,256,0,0,True,False


#### 4.2 Dependency Ordering

In [13]:
def build_migration_order(df_tables, df_fk):
    """"
    Build migration order using topology sort
    Input:
        df_tables: DataFrame with full_name of all tables
        df_fk: DataFrame with child_table and parent_table relationships
    Output:
        migration_order: list of table names for migration order
        self_ref_tables: list of table that references themselves
    """
    
    all_tables = df_tables['full_name'].tolist()
    
    self_ref = df_fk[
        df_fk['schema_name'] + '.' + df_fk['table_name'] == 
        df_fk['referenced_schema'] + '.' + df_fk['referenced_table']
    ]
    
    self_ref_tables = (
        self_ref['schema_name'] + '.' + self_ref['table_name']
    ).unique().tolist()
    
    fk_filtered = df_fk[
        df_fk['schema_name'] + '.' + df_fk['table_name'] != 
        df_fk['referenced_schema'] + '.' + df_fk['referenced_table']
    ]
    
    graph = defaultdict(set)
    in_degree = {t: 0 for t in all_tables}
    
    for _, row in fk_filtered.iterrows():
        child = f"{row['schema_name']}.{row['table_name']}"
        parent = f"{row['referenced_schema']}.{row['referenced_table']}"
        
        if child in in_degree and parent in in_degree:
            if child not in graph[parent]:
                graph[parent].add(child)
                in_degree[child] += 1
                
    queue = deque([t for t in all_tables if in_degree[t] == 0])
    migration_order = []
    
    while queue:
        table = queue.popleft()
        migration_order.append(table)
        
        for child in sorted(graph[table]):
            in_degree[child] -= 1
            if in_degree[child] == 0:
                queue.append(child)
                
    if len(migration_order) != len(all_tables):
        remaining = [t for t in all_tables if t not in migration_order]
        print(f"WARNING: Possible circular dependency in: {remaining}")
        
    return migration_order, self_ref_tables


try: 
    migration_order, self_ref_tables = build_migration_order(
        df_tables, df_fk
    )
    print(f"Migration order for {len(migration_order)} tables:\n")
    for i, table in enumerate(migration_order, 1):
       print(f"  {i:2d}. {table}")
    
    if self_ref_tables:
        print(f"\nSelf-referencing tables (add FK constraint AFTER load):")
        for t in self_ref_tables:
            print(f"  - {t}")
    
            
except Exception as e:
    print(f"Failed: {e}")        

Migration order for 71 tables:

   1. dbo.AWBuildVersion
   2. dbo.DatabaseLog
   3. dbo.ErrorLog
   4. HumanResources.Department
   5. HumanResources.Shift
   6. Person.AddressType
   7. Person.BusinessEntity
   8. Person.ContactType
   9. Person.CountryRegion
  10. Person.PhoneNumberType
  11. Production.Culture
  12. Production.Illustration
  13. Production.Location
  14. Production.ProductCategory
  15. Production.ProductDescription
  16. Production.ProductModel
  17. Production.ProductPhoto
  18. Production.ScrapReason
  19. Production.TransactionHistoryArchive
  20. Production.UnitMeasure
  21. Purchasing.ShipMethod
  22. Sales.CreditCard
  23. Sales.Currency
  24. Sales.SalesReason
  25. Sales.SpecialOffer
  26. Person.Person
  27. Purchasing.Vendor
  28. Sales.SalesTerritory
  29. Production.ProductSubcategory
  30. Production.ProductModelIllustration
  31. Production.ProductModelProductDescriptionCulture
  32. Sales.CountryRegionCurrency
  33. Sales.CurrencyRate
  34. HumanRes

### 5. Pre-Migration Quality Audit
##### 5.1 Present Null Check

In [14]:
def check_nulls(engine, df_dtypes):    
    def _check_table(engine, schema, table, columns):
        null_checks = ", ".join([
            f"SUM(CASE WHEN [{c}] IS NULL THEN 1 ELSE 0 END) AS [{c}_nulls]"
            for c in columns
        ])
        query = f"""
            SELECT COUNT(*) AS total_rows, {null_checks}
            FROM [{schema}].[{table}]
        """
        return pd.read_sql(query, engine)

    not_null_df = df_dtypes[df_dtypes['is_nullable'] == False]
    all_results = []

    for (schema, table), group in not_null_df.groupby(
        ['schema_name', 'table_name']
    ):
        try:
            result = _check_table(
                engine, schema, table,
                group['column_name'].tolist()
            )
            result.insert(0, 'table_name', f"{schema}.{table}")
            all_results.append(result)
        except Exception as e:
            print(f"Warning: Could not check {schema}.{table}: {e}")

    df = pd.concat(all_results, ignore_index=True)
    null_cols = [c for c in df.columns if c.endswith('_nulls')]
    df['has_null_issue'] = df[null_cols].sum(axis=1) > 0
    
    return df, df[df['has_null_issue']]

##### 5.2 Orphaned FK checks

In [15]:
def check_orphaned_fks(engine, df_fk):
    """
    Check child records whether they have corresponding parent.
    Input: df_fk from schema exploration
    """
    results = []
    
    for _, row in df_fk.drop_duplicates(
        subset=['schema_name', 'table_name', 
                'fk_column', 'referenced_schema',
                'referenced_table', 'referenced_column']
    ).iterrows():
        child_table = f"[{row['schema_name']}].[{row['table_name']}]"
        parent_table = f"[{row['referenced_schema']}].[{row['referenced_table']}]"
        child_col = f"[{row['fk_column']}]"
        parent_col = f"[{row['referenced_column']}]"
        
        query = f"""
            SELECT 
                '{row['schema_name']}.{row['table_name']}' AS child_table,
                '{row['fk_column']}' AS fk_column,
                '{row['referenced_schema']}.{row['referenced_table']}' AS parent_table,
                COUNT(*) AS orphaned_count
            FROM {child_table} c
            WHERE {child_col} IS NOT NULL
            AND NOT EXISTS (
                SELECT 1 FROM {parent_table} p
                WHERE p.{parent_col} = c.{child_col}
            )
        """
        try:
            result = pd.read_sql(query, engine)
            results.append(result)
        except Exception as e:
            print(f"Warning: Could not check {child_table}: {e}")
    
    if results:
        df = pd.concat(results, ignore_index=True)
        return df, df[df['orphaned_count'] > 0]
    return pd.DataFrame(), pd.DataFrame()

##### 5.3 Negative value checks

In [16]:
def check_negative_values(engine, df_dtypes):
    """
    Check numeric columns for negative values.
    Only check the columns that have names related to price/quantity/cost.
    """
    # Keywords cho business-relevant numeric columns
    keywords = ['price', 'cost', 'quantity', 'amount', 
                'rate', 'salary', 'stock', 'inventory']
    
    numeric_types = ['int', 'bigint', 'smallint', 'decimal', 
                     'numeric', 'float', 'money']
    
    # Filter relevant columns
    candidates = df_dtypes[
        df_dtypes['data_type'].isin(numeric_types) &
        df_dtypes['column_name'].str.lower().str.contains(
            '|'.join(keywords), na=False
        )
    ]
    
    results = []
    for (schema, table), group in candidates.groupby(
        ['schema_name', 'table_name']
    ):
        neg_checks = ", ".join([
            f"SUM(CASE WHEN [{c}] < 0 THEN 1 ELSE 0 END) AS [{c}_negatives]"
            for c in group['column_name'].tolist()
        ])
        query = f"""
            SELECT 
                '{schema}.{table}' AS table_name,
                COUNT(*) AS total_rows,
                {neg_checks}
            FROM [{schema}].[{table}]
        """
        try:
            result = pd.read_sql(query, engine)
            results.append(result)
        except Exception as e:
            print(f"Warning: {schema}.{table}: {e}")
    
    if results:
        df = pd.concat(results, ignore_index=True)
        neg_cols = [c for c in df.columns if c.endswith('_negatives')]
        df['has_negative'] = df[neg_cols].sum(axis=1) > 0
        return df, df[df['has_negative']]
    return pd.DataFrame(), pd.DataFrame()

##### 5.4 Future date checks

In [17]:
def check_future_dates(engine, df_dtypes):
    """
    Check date/datetime columns for future date values.
    """
    date_types = ['date', 'datetime', 'datetime2', 'smalldatetime']
    
    date_cols = df_dtypes[df_dtypes['data_type'].isin(date_types)]
    
    results = []
    for (schema, table), group in date_cols.groupby(
        ['schema_name', 'table_name']
    ):
        future_checks = ", ".join([
            f"SUM(CASE WHEN [{c}] > GETDATE() THEN 1 ELSE 0 END) AS [{c}_future]"
            for c in group['column_name'].tolist()
        ])
        query = f"""
            SELECT 
                '{schema}.{table}' AS table_name,
                COUNT(*) AS total_rows,
                {future_checks}
            FROM [{schema}].[{table}]
        """
        try:
            result = pd.read_sql(query, engine)
            results.append(result)
        except Exception as e:
            print(f"Warning: {schema}.{table}: {e}")
    
    if results:
        df = pd.concat(results, ignore_index=True)
        future_cols = [c for c in df.columns if c.endswith('_future')]
        df['has_future_date'] = df[future_cols].sum(axis=1) > 0
        return df, df[df['has_future_date']]
    return pd.DataFrame(), pd.DataFrame()

##### 5.5 Summary

In [18]:
try:
    warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
    
    print("=" * 50)
    print("PRE-MIGRATION QUALITY CHECKS — SUMMARY")
    print("=" * 50)

    df_null_checks, null_issues = check_nulls(
        sql_connection_engine, df_dtypes
    )
    _, fk_issues = check_orphaned_fks(
        sql_connection_engine, df_fk
    )
    _, neg_issues = check_negative_values(
        sql_connection_engine, df_dtypes
    )
    df_date_checks, date_issues = check_future_dates(
        sql_connection_engine, df_dtypes
    )

    results = {
        'Null checks':           len(null_issues),
        'Orphaned FK checks':    len(fk_issues),
        'Negative value checks': len(neg_issues),
        'Future date checks':    len(date_issues)
    }

    for check, count in results.items():
        status = "✓ PASSED" if count == 0 else f"⚠ {count} issue(s)"
        print(f"{check:<25} {status}")

    total = sum(results.values())
    print("=" * 50)
    if total == 0:
        print("✓ All checks passed. Ready to migrate.")
    else:
        print(f"⚠ {total} total issue(s) found. Review before migrating.")



except Exception as e:
    print(f"Summary failed: {e}")

PRE-MIGRATION QUALITY CHECKS — SUMMARY


Null checks               ✓ PASSED
Orphaned FK checks        ✓ PASSED
Negative value checks     ✓ PASSED
Future date checks        ⚠ 3 issue(s)
⚠ 3 total issue(s) found. Review before migrating.
